# 03 · Model Training

**Goal:** compare models, tune the best one (LightGBM), choose a decision threshold and save it.

**Input:** `data/hotel_bookings_clean.csv` · **Output:** `models/lgbm_model.pkl`, `feature_columns.pkl`, `threshold.pkl`

> Feature engineering from notebook 02 is repeated here so this notebook runs on its own.
> The same logic lives in `src/predictor.py` for inference.

In [1]:
import os
import joblib
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import (RandomizedSearchCV, StratifiedKFold,
                                     cross_val_predict, cross_val_score, train_test_split)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

RANDOM_STATE = 42

## 1. Prepare features

In [2]:
df = pd.read_csv("../data/hotel_bookings_clean.csv")

df["total_nights"] = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
df["total_guests"] = df["adults"] + df["children"] + df["babies"]
df["has_kids"] = ((df["children"] + df["babies"]) > 0).astype(int)
df["has_agent"] = (df["agent"] != 0).astype(int)
df["ever_canceled_before"] = (df["previous_cancellations"] > 0).astype(int)

months = ["January", "February", "March", "April", "May", "June", "July",
          "August", "September", "October", "November", "December"]
df["arrival_month"] = df["arrival_date_month"].map({m: i + 1 for i, m in enumerate(months)})

top_countries = df["country"].value_counts().head(10).index
df["country"] = df["country"].where(df["country"].isin(top_countries), "Other")

df = df.drop(columns=["arrival_date_month", "arrival_date_year", "agent", "assigned_room_type"])

## 2. Encode and split (80% train / 20% test, stratified)

In [3]:
X = df.drop(columns=["is_canceled"])
y = df["is_canceled"]

cat_cols = X.select_dtypes(include=["object", "string"]).columns
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(X_train.shape, X_test.shape)

(95360, 58) (23841, 58)


## 3. Compare models (5-fold cross-validation, ROC-AUC)

Only the training set is used, so the test set stays untouched for the final check.

In [4]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

models = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "Decision Tree": DecisionTreeClassifier(max_depth=10, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE),
    "XGBoost": XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                             random_state=RANDOM_STATE, eval_metric="logloss"),
    "LightGBM": LGBMClassifier(n_estimators=300, random_state=RANDOM_STATE, verbose=-1),
}

for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1)
    print(f"{name:20s} ROC-AUC: {scores.mean():.4f} +/- {scores.std():.4f}")

Logistic Regression  ROC-AUC: 0.8871 +/- 0.0016
Decision Tree        ROC-AUC: 0.9093 +/- 0.0020
Random Forest        ROC-AUC: 0.9494 +/- 0.0014
XGBoost              ROC-AUC: 0.9395 +/- 0.0015
LightGBM             ROC-AUC: 0.9411 +/- 0.0019


## 4. Tune LightGBM (randomized search)

In [5]:
param_grid = {
    "n_estimators": [300, 500, 800],
    "learning_rate": [0.03, 0.05, 0.1],
    "num_leaves": [31, 63, 127, 255],
    "max_depth": [-1, 10, 15],
    "min_child_samples": [10, 20, 50],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.7, 0.85, 1.0],
}

search = RandomizedSearchCV(
    LGBMClassifier(random_state=RANDOM_STATE, verbose=-1, subsample_freq=1),
    param_grid, n_iter=25, cv=cv, scoring="roc_auc", n_jobs=-1, random_state=RANDOM_STATE,
)
search.fit(X_train, y_train)

print("Best CV ROC-AUC:", round(search.best_score_, 4))
print("Best params:", search.best_params_)
best_lgbm = search.best_estimator_

Best CV ROC-AUC: 0.9513
Best params: {'subsample': 0.8, 'num_leaves': 255, 'n_estimators': 800, 'min_child_samples': 20, 'max_depth': 15, 'learning_rate': 0.05, 'colsample_bytree': 0.85}


## 5. Choose the decision threshold

A missed cancellation (false negative) costs more than a false alarm, so we lower the threshold below 0.5.
It is chosen on **cross-validated training predictions** to avoid tuning on the test set.

In [6]:
train_prob = cross_val_predict(best_lgbm, X_train, y_train, cv=cv,
                               method="predict_proba", n_jobs=-1)[:, 1]

for t in [0.5, 0.45, 0.4, 0.35]:
    cm = confusion_matrix(y_train, (train_prob >= t).astype(int))
    print(f"Threshold {t}: recall={cm[1, 1] / cm[1].sum():.2f}, precision={cm[1, 1] / cm[:, 1].sum():.2f}")

Threshold 0.5: recall=0.81, precision=0.86
Threshold 0.45: recall=0.83, precision=0.85
Threshold 0.4: recall=0.84, precision=0.83
Threshold 0.35: recall=0.86, precision=0.81


## 6. Final evaluation on the test set

In [7]:
THRESHOLD = 0.45

best_lgbm.fit(X_train, y_train)
test_prob = best_lgbm.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= THRESHOLD).astype(int)

print(classification_report(y_test, test_pred, target_names=["Not Canceled", "Canceled"]))
print("Test ROC-AUC:", round(roc_auc_score(y_test, test_prob), 4))
print(confusion_matrix(y_test, test_pred))

              precision    recall  f1-score   support

Not Canceled       0.91      0.91      0.91     15002
    Canceled       0.84      0.84      0.84      8839

    accuracy                           0.88     23841
   macro avg       0.88      0.88      0.88     23841
weighted avg       0.88      0.88      0.88     23841

Test ROC-AUC: 0.9541
[[13636  1366]
 [ 1395  7444]]


## 7. Save artifacts

In [8]:
os.makedirs("../models", exist_ok=True)
joblib.dump(best_lgbm, "../models/lgbm_model.pkl")
joblib.dump(list(X_train.columns), "../models/feature_columns.pkl")
joblib.dump(THRESHOLD, "../models/threshold.pkl")
print("Saved model, feature columns and threshold to ../models/")

Saved model, feature columns and threshold to ../models/
